# Descargar y ejecutar un modelo de Hugging Face **en local**

A diferencia de la notebook anterior (que usaba la caché por defecto), acá **descargamos explícitamente** los archivos del modelo a una carpeta del proyecto (`./models/`) y después lo cargamos desde ahí, sin volver a tocar internet.

Modelo: **`google/flan-t5-small`** — modelo abierto tipo *encoder-decoder* (seq2seq) afinado para seguir instrucciones (traducción, preguntas, etc.).

Esto es útil para entornos sin internet en tiempo de ejecución, o para versionar/compartir un modelo concreto.

## Instalación de dependencias
En local ya están en `requirements.txt`. En Colab, descomentá la celda siguiente.

In [1]:
# !pip install -q transformers torch huggingface_hub

## 1. Descargar el modelo a una carpeta local

In [2]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = 'google/flan-t5-small'
LOCAL_DIR = Path('models') / 'flan-t5-small'

# Descarga todos los archivos del modelo al directorio local.
# allow_patterns evita bajar formatos que no usamos (por ejemplo, pesos de TF/Flax).
ruta = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(LOCAL_DIR),
    allow_patterns=['*.json', '*.model', '*.txt', '*.safetensors', 'spiece.model'],
)
print('Modelo descargado en:', ruta)

C:\Users\02407\Documents\itba\curso-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo descargado en: C:\Users\02407\Documents\itba\curso-llm\models\flan-t5-small


## 2. Ver qué archivos se descargaron

In [3]:
import os
for f in sorted(os.listdir(LOCAL_DIR)):
    tam = os.path.getsize(LOCAL_DIR / f) / 1e6
    print(f'{f:35s} {tam:8.2f} MB')

.cache                                  0.00 MB
config.json                             0.00 MB
generation_config.json                  0.00 MB
model.safetensors                     307.87 MB
special_tokens_map.json                 0.00 MB
spiece.model                            0.79 MB
tokenizer.json                          2.42 MB
tokenizer_config.json                   0.00 MB


## 3. Cargar el modelo **desde la carpeta local**

Usamos `local_files_only=True` para asegurarnos de que NO se descarga nada: todo sale de `./models/flan-t5-small`.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_DIR, local_files_only=True)
model.eval()
print('Modelo cargado desde disco local. Parametros:', sum(p.numel() for p in model.parameters()))

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo cargado desde disco local. Parametros: 76961152


## 4. Probar el modelo con algunas instrucciones

In [5]:
def generar(prompt: str, max_new_tokens: int = 40) -> str:
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompts = [
    'translate English to German: How old are you?',
    'Question: What is the capital of France? Answer:',
    'Summarize: The course covers tokenization, embeddings and attention in modern LLMs.',
]

for p in prompts:
    print('>>', p)
    print('  ->', generar(p))
    print()

>> translate English to German: How old are you?


  -> Wie ich er bitten?

>> Question: What is the capital of France? Answer:
  -> london

>> Summarize: The course covers tokenization, embeddings and attention in modern LLMs.


  -> The course covers tokenization, embeddings and attention in modern LLMs.



## 5. Notas

- La carpeta `models/` está en `.gitignore`: **no** se sube al repositorio (los pesos pesan y no conviene versionarlos).
- Para volver a correr sin internet, basta con tener la carpeta `models/flan-t5-small` y usar `local_files_only=True`.
- Probá cambiar `MODEL_ID` por otro modelo abierto (por ejemplo `google/flan-t5-base`) y compará los resultados. ✅